# Lost in the Middle — Qwen 2.5 7B | Position 29 ONLY
**Model:** qwen2.5:7b-instruct | All 2,655 questions | 30 docs | **Position 29 only**  
**GPU:** 2x T4 | Estimated time: ~2.5–3 hours  

Previous results for positions 0, 4, 9, 14, 19, 24 are already baked in.
This notebook only runs Position 29 and produces the final complete JSON + graph.

**Evaluation follows the original paper exactly:**
- Response truncated at first `\n` before matching
- All valid answer synonyms checked (`data["answers"]`)
- `normalize_answer`: lowercase → remove articles → remove punctuation → fix whitespace
- temperature=0 (greedy decoding), full 2,655 questions
- Concise answer prompt (keeps inference fast: ~2–3s per question)

In [ ]:
!pip install ollama

In [ ]:
import subprocess
import time

print("Installing dependencies...")
subprocess.run("apt-get update && apt-get install -y zstd", shell=True, check=True)
subprocess.run("pip install ollama==0.6.2", shell=True, check=True)

print("Installing Ollama...")
subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, check=True)

print("Starting server...")
subprocess.Popen("ollama serve", shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(10)

print("Pulling model...")
subprocess.run("ollama pull qwen2.5:7b-instruct", shell=True, check=True)
print("Ollama ready!")

In [ ]:
import os
import json
import string
import re
import time
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import ollama

# =====================================================
# CONFIG
# =====================================================
MODEL_NAME = "qwen2.5:7b-instruct"
NUM_CTX = 12288
DATASET_PATH = "/kaggle/input/datasets/arinjaysarkar476/lost-in-the-middle"
OUTPUT_DIR = "/kaggle/working"

folder = "30_total_documents"
target_pos = 29

# =====================================================
# PREVIOUS COMPLETED RESULTS (positions 0, 4, 9, 14, 19, 24)
# =====================================================
results_data = [
    {"Context Size (Documents)": "30", "Position of Answer (Gold Index)": 0,  "Accuracy (%)": 54.80226, "Correct": 1455, "Total": 2655},
    {"Context Size (Documents)": "30", "Position of Answer (Gold Index)": 4,  "Accuracy (%)": 47.68362, "Correct": 1266, "Total": 2655},
    {"Context Size (Documents)": "30", "Position of Answer (Gold Index)": 9,  "Accuracy (%)": 47.34463, "Correct": 1257, "Total": 2655},
    {"Context Size (Documents)": "30", "Position of Answer (Gold Index)": 14, "Accuracy (%)": 47.57062, "Correct": 1263, "Total": 2655},
    {"Context Size (Documents)": "30", "Position of Answer (Gold Index)": 19, "Accuracy (%)": 49.03955, "Correct": 1302, "Total": 2655},
    {"Context Size (Documents)": "30", "Position of Answer (Gold Index)": 24, "Accuracy (%)": 49.56685, "Correct": 1316, "Total": 2655},
]

# =====================================================
# NORMALIZATION (paper-exact)
# =====================================================
def normalize_answer(s: str) -> str:
    def remove_articles(text):
        return re.sub(r"\b(a|an|the)\b", " ", text)
    def white_space_fix(text):
        return " ".join(text.split())
    def remove_punc(text):
        exclude = set(string.punctuation)
        return "".join(ch for ch in text if ch not in exclude)
    def lower(text):
        return text.lower()
    return white_space_fix(remove_articles(remove_punc(lower(s))))

# =====================================================
# LOAD / RESUME SAVED RESPONSES
# =====================================================
responses_file = os.path.join(OUTPUT_DIR, "all_responses_qwen25_30docs_pos29.json")
results_file   = os.path.join(OUTPUT_DIR, "results_qwen25_30docs_complete.json")

if os.path.exists(responses_file):
    with open(responses_file, "r") as f:
        all_responses = json.load(f)
    print(f"Loaded {len(all_responses)} saved responses — resuming!")
else:
    all_responses = []
    print("No saved responses found — starting fresh.")

# Resume point
position_responses = [r for r in all_responses if r["position"] == target_pos]
if position_responses:
    start_q = max(r["q_idx"] for r in position_responses) + 1
    correct_matches = sum(1 for r in position_responses if r["match"])
else:
    start_q = 0
    correct_matches = 0

print(f"Position {target_pos} | Starting from Q{start_q} | Correct so far: {correct_matches}")

# =====================================================
# DATASET FILE
# =====================================================
file_path = os.path.join(
    DATASET_PATH, "qa_data", folder,
    f"nq-open-{folder}_gold_at_{target_pos}.jsonl"
)
print(f"Loading: {file_path}")

with open(file_path, "r", encoding="utf-8") as f:
    lines = f.readlines()

total_questions = len(lines)
pos_start = time.time()

print(f"Total questions: {total_questions}")
print("=" * 60)

# =====================================================
# EVALUATION LOOP
# =====================================================
for q_idx in range(start_q, total_questions):

    data = json.loads(lines[q_idx])
    target_query   = data["question"]
    target_answers = data["answers"]
    ctxs           = data["ctxs"]

    context_parts = []
    for j, ctx in enumerate(ctxs):
        context_parts.append(f"Document [{j+1}](Title: {ctx['title']}) {ctx['text']}")
    context_str = "\n".join(context_parts)

    # --- Same fast, concise prompt as the official notebooks ---
    prompt = (
        "Write a high-quality answer for the given question using only the provided search results "
        "(some of which might be irrelevant).\n\n"
        f"{context_str}\n\n"
        f"Question: {target_query}\n"
        "Answer: (Keep the answer as concise as possible, preferably one or two words. "
        "Do not write full sentences.)"
    )

    try:
        q_start = time.time()
        response = ollama.chat(
            model=MODEL_NAME,
            messages=[{"role": "user", "content": prompt}],
            options={"num_ctx": NUM_CTX, "temperature": 0.0}
        )
        q_time = time.time() - q_start

        raw_output = response["message"]["content"]
        output = raw_output.split("\n")[0].strip()           # first-line truncation
        normalized_prediction = normalize_answer(output)

        success = any(
            normalize_answer(ans) in normalized_prediction
            for ans in target_answers
        )
        if success:
            correct_matches += 1

        all_responses.append({
            "folder": folder,
            "position": target_pos,
            "q_idx": q_idx,
            "question": target_query,
            "target_answers": target_answers,
            "raw_output": raw_output.strip(),
            "truncated_output": output,
            "match": success,
            "time_sec": round(q_time, 2)
        })

        # Save every 50 questions
        if (q_idx + 1) % 50 == 0 or q_idx == total_questions - 1:
            with open(responses_file, "w") as f:
                json.dump(all_responses, f)
            acc_so_far = (correct_matches / (q_idx - start_q + 1)) * 100
            elapsed = time.time() - pos_start
            eta = (elapsed / (q_idx - start_q + 1)) * (total_questions - q_idx - 1)
            print(
                f"[Q{q_idx+1}/{total_questions}] ({q_time:.1f}s) "
                f"Acc: {acc_so_far:.1f}% | "
                f"ETA: {eta/60:.0f}min | "
                f"{target_answers} -> {output[:40]} | {success}"
            )

    except Exception as e:
        print(f"Error at Q{q_idx+1}: {e}")
        all_responses.append({
            "folder": folder, "position": target_pos, "q_idx": q_idx,
            "question": target_query, "target_answers": target_answers,
            "raw_output": f"ERROR: {str(e)}",
            "truncated_output": "", "match": False, "time_sec": 0
        })

# =====================================================
# FINAL RESULT
# =====================================================
total_pos_responses = [r for r in all_responses if r["position"] == target_pos]
final_correct = sum(1 for r in total_pos_responses if r["match"])
accuracy = (final_correct / total_questions) * 100
pos_time = time.time() - pos_start

print(f"\n{'='*60}")
print(f"Position {target_pos} DONE: {accuracy:.2f}% ({final_correct}/{total_questions}) [{pos_time/60:.1f}min]")
print(f"{'='*60}")

# Append position 29 result
results_data.append({
    "Context Size (Documents)": "30",
    "Position of Answer (Gold Index)": 29,
    "Accuracy (%)": round(accuracy, 5),
    "Correct": final_correct,
    "Total": total_questions
})

# Save complete results
with open(results_file, "w") as f:
    json.dump(results_data, f, indent=2)

print(f"\nComplete results saved to: {results_file}")
print(json.dumps(results_data, indent=2))

In [ ]:
# =====================================================
# FINAL GRAPH — All 7 positions
# =====================================================
df = pd.DataFrame(results_data)
df = df.sort_values("Position of Answer (Gold Index)")

sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Full scale (0-100)
sns.lineplot(ax=axes[0], data=df,
             x="Position of Answer (Gold Index)", y="Accuracy (%)",
             marker="o", color="#2196F3", linewidth=2.5)
axes[0].set_title(f"Lost in the Middle — {MODEL_NAME} (30 docs)", fontsize=13)
axes[0].set_ylim(0, 100)
axes[0].set_xticks(df["Position of Answer (Gold Index)"].unique())

# Zoomed
margin = 3
y_min = df["Accuracy (%)"].min() - margin
y_max = df["Accuracy (%)"].max() + margin
sns.lineplot(ax=axes[1], data=df,
             x="Position of Answer (Gold Index)", y="Accuracy (%)",
             marker="o", color="#E91E63", linewidth=2.5)
axes[1].set_title(f"Zoomed — {MODEL_NAME} (30 docs)", fontsize=13)
axes[1].set_ylim(y_min, y_max)
axes[1].set_xticks(df["Position of Answer (Gold Index)"].unique())

for ax in axes:
    for x, y in zip(df["Position of Answer (Gold Index)"], df["Accuracy (%)"]):
        ax.annotate(f"{y:.1f}%", (x, y), textcoords="offset points",
                    xytext=(0, 8), ha="center", fontsize=9)

plt.tight_layout()
graph_file = os.path.join(OUTPUT_DIR, "qwen25_30docs_complete_graph.png")
plt.savefig(graph_file, dpi=300)
plt.show()

print(f"Graph saved: {graph_file}")
print("\nFinal Summary:")
print(df[["Position of Answer (Gold Index)", "Accuracy (%)", "Correct", "Total"]].to_string(index=False))